# Building the signal service

### Everything on this screen is running for real. Nothing here is a slide.

---

You are going to watch a data platform break in the way that costs companies the
most money: **quietly**. No crash, no error, no alert, every dashboard green, and
a number that a real team makes decisions with is wrong.

Then you are going to **build the service that catches it**, from nothing, one
idea at a time. Not read it. Build it, run it, and only at the end compare what
you built with what is in the repository.

**No prior knowledge assumed.** If you have never written a data pipeline, you
are in the right room.

In [ ]:
import sys; sys.path.insert(0, '..')
import inspect, json, statistics, psycopg
from pipelines.lib.config import dsn, SCHEMA, MONGO_URI
from nb import show, sql, fetch, run

print('connected to the warehouse.')

### Start from a clean slate

Run this if anything here has been run before, or if a previous session left the
estate half broken. It puts the source data back, empties the warehouse, forgets
every incident, and rebuilds all eight pipelines from nothing.

**Safe to run at any point.** If you get lost later, come back to this cell.

In [ ]:
run('break_it.py', '--fix')     # undo the break, if it is still in place
run('cli.py', 'reset')          # empty warehouse, no incidents, no artifacts
run('cli.py', 'run', 'all')     # rebuild all eight pipelines, about 40 seconds

---

# Part 1 · A number goes wrong and nothing fails

## The company

KERB is a ride hailing company. Riders book cars, drivers accept them, money
moves. The data lives in **six different systems**, because it was built by six
different teams over six different years:

| where | what is in it | what it looks like |
|---|---|---|
| PostgreSQL | the rides themselves | rows in a table |
| Kafka | every event in a ride's life | messages on a stream |
| **MongoDB** | **what the driver's phone reported** | **documents, no fixed shape** |
| a partner's API | whether the money settled | JSON over HTTP |
| MinIO | the regulator's nightly file | gzipped CSV |
| PostgreSQL | the list of zones | a small reference table |

Nobody can answer *"how much did we earn on Tuesday"* from six systems at once.
So we copy all six into one place, shaped for questions instead of for apps.
That place is the **warehouse**, and this is what finance reads every morning.

In [ ]:
sql(f"""
    SELECT trip_date, rides, completed, cancelled, revenue, avg_surge
    FROM {SCHEMA}.gold_daily ORDER BY trip_date DESC LIMIT 7
""", 'gold_daily · the answer somebody reads every morning')

Seven days, one row each. Note **`avg_surge`** on the right. Surge is the
multiplier applied when demand is high, and the pricing team use it to decide
whether the pricing model is behaving.

**Remember that column.** It is the one that is about to go wrong.

## Where that number comes from

It is not typed in by anybody. It is carried, one hop at a time, from a phone in
a car to that table.

```
    the driver's phone
          |
          v
    MongoDB   kerb_app.driver_app_events        <- a document per event
          |
          |   p3_bronze_driver_app      copy it, keep it faithful
          v
    teach.bronze_driver_app                     <- our copy
          |
          |   p7_silver_rides           one clean row per ride
          v
    teach.silver_rides
          |
          |   p8_gold_daily             one row per day
          v
    teach.gold_daily                            <- what finance reads
```

Four hops. **The break happens at the very first one**, and every hop after it
carries on working perfectly.

## What one of those documents actually looks like

Not a diagram of a document. A real one, read out of MongoDB right now, printed
exactly as it is stored.

In [ ]:
from pymongo import MongoClient

events = MongoClient(MONGO_URI).kerb_app.driver_app_events

def one_document(version):
    """One real trip_offer document from a given app version."""
    return events.find_one({'app.version': version, 'event_type': 'trip_offer'},
                           {'_id': 0, 'location': 0, 'device': 0})

print(json.dumps(one_document('4.2.0'), indent=2, default=str))

Read the bottom of that document. The surge multiplier is in there, nested:

```
    "payload": { "pricing": { "surgeFactor": 1.07 } }
```

**Nothing enforces that shape.** MongoDB accepts a document of any shape at all.
The phone decides where to put the value, and the phone is written by a different
team, on a different release cycle, who have never met you.

## So the pipeline has to know where to look

Here is the real code that reads surge out of a document. Not a description of
it, the code, out of the file that ran to build the table above.

In [ ]:
from pipelines import p3_bronze_driver_app as p3

print('the four places this pipeline knows to look:')
print()
for path in p3.SURGE_PATHS:
    print(f'    doc[{"][".join(repr(k) for k in path)}]')
print()
print('and the function that tries them, in order:')
print()
print(inspect.getsource(p3.surge_of))

Four known places, tried in order. The first that has a value wins.

Those four paths are not a design. They are **scar tissue**: each was added the
day a release moved the field and somebody spent an afternoon finding out.

And look at what happens when none of them match. It does **not** write a zero
and it does **not** write a null. It **holds** the document, with the reason
attached. That decision matters in about two minutes.

---

## Now break it

A new mobile release goes out. It moves the surge value one level deeper, from
`payload.pricing.surgeFactor` to `payload.pricing.surge.factor`.

A perfectly reasonable refactor. They are tidying their own payload. Nobody tells
the data team, because from where they sit nothing about the app changed.

In [ ]:
run('break_it.py')

## The same document, again

Same query, same collection, same app version. This is what the phone sends now.

In [ ]:
print(json.dumps(one_document('4.2.0'), indent=2, default=str))

Put the two side by side, because this one change is the whole lesson.

| | where surge lives |
|---|---|
| **before** | `payload` &rarr; `pricing` &rarr; **`surgeFactor`** |
| **after** | `payload` &rarr; `pricing` &rarr; **`surge`** &rarr; **`factor`** |

The value is still there. It is still correct. It is still 1.07. It is one level
deeper and spelled differently, and **none of our four known paths matches it**.

Nothing about this is malicious or even careless. This is Tuesday.

## Re-run the pipelines and watch nothing fail

In [ ]:
run('cli.py', 'run', 'all')

## Read that output slowly, because this is the part people miss

**Every pipeline says `ok`.** Nothing crashed. Nothing raised. No retry, no
alert, no red anything.

But look at the driver app line:

```
    p3_bronze_driver_app: ok   read 40,000   wrote 30,131
```

It read forty thousand documents and wrote thirty thousand. **Nine thousand
eight hundred and sixty nine documents did not become rows**, and it still
reported success, because from its point of view nothing went wrong: it was
asked to copy what it could read, and it did exactly that.

In [ ]:
sql(f"""
    SELECT pipeline, status, rows_in, rows_out,
           rows_in - rows_out AS did_not_land,
           to_char(started_at, 'HH24:MI:SS') AS at
    FROM {SCHEMA}.runs
    WHERE pipeline = 'p3_bronze_driver_app'
    ORDER BY started_at DESC LIMIT 3
""", 'the run log · success, and ten thousand rows short')

## Where did the missing ten thousand go?

They were **held**. Not dropped, not defaulted, not silently zeroed: written to
a quarantine table with the reason attached and the original document intact.

In [ ]:
sql(f"""
    SELECT reason, count(*) AS records
    FROM {SCHEMA}.quarantine
    WHERE pipeline = 'p3_bronze_driver_app'
    GROUP BY 1
""", 'why they were held')

with psycopg.connect(dsn()) as c:
    held = c.execute(f"""SELECT payload FROM {SCHEMA}.quarantine
                         WHERE pipeline = 'p3_bronze_driver_app'
                         ORDER BY seen_at DESC LIMIT 1""").fetchone()[0]

print('one held document, exactly as it arrived:')
print()
print(json.dumps(held, indent=2)[:700])

There it is. `payload.pricing.surge.factor`, sitting in quarantine, value intact.

**The evidence of what broke is in the warehouse**, written by the pipeline at
the moment it happened, which is the only time anybody knew enough to write it
down.

> Holding a record costs you an incomplete table for a day.
> Defaulting it to zero costs you a wrong number forever, with nothing to point at.

## What the warehouse says now

The whole 4.2.0 release has **vanished** from our copy. Not corrupted. Gone.

In [ ]:
sql(f"""
    SELECT app_version, count(*) AS rows_that_landed
    FROM {SCHEMA}.bronze_driver_app
    GROUP BY 1 ORDER BY 1
""", 'bronze_driver_app · one app version is missing entirely')

## This is the entire problem

| What a normal platform checks | What it says today |
|---|---|
| did the pipeline run? | yes |
| did it fail? | no |
| did it throw an error? | no |
| did every row land? | **no, and nothing asked** |
| is the number correct? | **no** |

> **A pipeline that fails wakes somebody up.**
>
> **A pipeline that succeeds while quietly carrying less data than it should
> wakes nobody.**

Every check in the left column stays green. So does every error log, every retry
policy, and every dashboard that shows pipeline status, because all of them watch
**the machinery** and none of them watch **the number**.

**So we are going to build the thing that watches the number.** From nothing.

---

# Part 2 · Building the signal service, step by step

Twelve steps. Each one is a decision, and each one exists because the previous
step was not enough. Nothing here is imported until we have written it.

## Step 1 · A number, on its own, is useless

Here is a real number, correctly calculated, from the table finance reads.

In [ ]:
with psycopg.connect(dsn()) as c:
    revenue = float(c.execute(
        f'SELECT revenue FROM {SCHEMA}.gold_daily ORDER BY trip_date DESC LIMIT 1'
    ).fetchone()[0])

print(f'revenue yesterday: {revenue:,.2f}')
print()
print('Question for the room: is that good or bad?')

Nobody can answer that. Not you, not me, not any alerting product ever sold.

> **A number has no opinion about itself.**

To have an opinion you need a second thing: **what it normally is**.

In [ ]:
with psycopg.connect(dsn()) as c:
    history = [float(r[0]) for r in c.execute(
        f'SELECT revenue FROM {SCHEMA}.gold_daily ORDER BY trip_date DESC OFFSET 1')]

baseline = statistics.mean(history)
spread   = statistics.pstdev(history)

print(f'  yesterday      {revenue:>14,.2f}')
print(f'  normally       {baseline:>14,.2f}    over {len(history)} days')
print(f'  usual wobble   {spread:>14,.2f}')
print()
print(f'  {(revenue - baseline) / max(spread, 0.01):>5.2f} wobbles from normal')

That last number has a name. It is a **z-score**: how many standard deviations
from the mean. It is the whole of the statistics in this project.

> ### A **KPI** is a number with a definition and an owner. It can never raise an alarm.
> ### A **signal** is a KPI plus a baseline plus a tolerance. That is the thing that goes off.

Most "we need better monitoring" conversations are actually about the missing
second half, and nobody wants to write it, because writing it means committing in
public to a sentence like *"revenue is normally about 1.4 million a day"*.

## Step 2 · Give the number a definition and an owner

We could keep this in a dictionary. We are not going to, because six months from
now somebody will ask *"who owns this and what does it mean"* and a dictionary
has no answer.

In [ ]:
from dataclasses import dataclass, field

@dataclass(frozen=True)
class MyKPI:
    name: str          # what people call it in Slack
    title: str         # the same thing, in words
    owner: str         # a team that exists, not 'data'
    watches: str       # the table it reads
    means: str         # written FOR the owner, who has never read this code
    sql: str           # returns exactly one row, one column
    unit: str = ''

surge = MyKPI(
    name='surge_coverage_pct',
    title='Share of driver app records carrying a surge value',
    owner='pricing',
    watches=f'{SCHEMA}.bronze_driver_app',
    means='What share of driver app records arrived with a surge value we could '
          'read. This moves when the mobile team renames a field.',
    unit='%',
    sql=f"""SELECT coalesce((SELECT round(100.0 * rows_out / nullif(rows_in, 0), 2)
                             FROM {SCHEMA}.runs
                             WHERE pipeline = 'p3_bronze_driver_app'
                               AND status = 'success'
                             ORDER BY started_at DESC LIMIT 1), 100)::float""")

print(surge.name, '·', surge.owner)
print(surge.means)

### Three things to notice, and all three are about people

**`owner` is a team, not "data".** A signal nobody owns is a signal nobody acts
on. If you cannot name the team, you have not finished defining the KPI.

**`means` is written for the owner**, who was asleep and has never read this
code. This ends up in the ticket, the page, and the terminal.

**`sql` measures what the pipeline was OFFERED, not what it wrote.** Look at that
query again: it reads `rows_in` and `rows_out` from the run log.

The obvious version, `count(surge) / count(*)` over the landed table, is **100%
by construction** and can never move, because p3 never writes a null surge: a
document it cannot read is held. A KPI that cannot detect the failure it is named
after is worse than none, because it is reassuring.

## Step 3 · Read it, and never let it raise

Reading a number sounds like one line. It is one line and one decision.

In [ ]:
def read(kpi):
    """Run the query. Return the number, or the reason there isn't one."""
    try:
        with psycopg.connect(dsn()) as c:
            row = c.execute(kpi.sql).fetchone()
        return {'value': float(row[0]) if row and row[0] is not None else None,
                'error': None}
    except Exception as e:
        # A KPI that cannot be measured is NOT a KPI that is fine.
        return {'value': None, 'error': f'{type(e).__name__}: {e}'}

print(read(surge))

> **"could not be measured" and "was measured and it is wrong" are different
> outcomes, and only one of them means somebody typed a bad query.**

Collapse them into `0` and a dropped table looks exactly like a healthy day.

There is a second reason for the `try`, and it cost real time to find. Postgres
aborts **the whole transaction** on any error. Thirteen KPIs sharing one
connection, one of them with a typo, and the other twelve silently return nothing.
One poison KPI must not be able to silence the board.

## Step 4 · Judge it, separately

Reading and judging are different jobs, so they are different functions.

In [ ]:
def judge(kpi, reading, baseline, spread=None, tolerance=0.0, z_threshold=4.0):
    """Compare the number to what it should be. Returns a verdict, never raises."""
    if reading['value'] is None:
        return {'breached': False, 'detail': reading['error'] or 'not measured'}

    value = reading['value']

    if spread is None:                       # baseline is A RULE somebody decided
        off = abs(value - baseline)
        return {'breached': off > tolerance, 'z': None,
                'detail': f'{off:.2f} away from the rule of {baseline:g}, '
                          f'{tolerance:g} allowed'}

    z = (value - baseline) / max(spread, 1e-9)     # baseline is FROM HISTORY
    return {'breached': abs(z) > z_threshold, 'z': round(z, 2),
            'detail': f'{z:+.2f} standard deviations from {baseline:,.2f}'}

print('rule    ', judge(surge, read(surge), baseline=100.0, tolerance=1.0))
print('history ', judge(surge, read(surge), baseline=99.0, spread=0.4))

### Why they are two functions and not one

Because you will want to read a number without judging it (a chart), and judge a
number you already have (a backfill, a test). Fused together you can do neither,
and every test needs a database.

### The two kinds of normal, and both are correct

| | where the baseline comes from | breached when | example |
|---|---|---|---|
| **from history** | the last N readings of this same number | z-score past a threshold | revenue per day |
| **a rule** | a person decided it | past a tolerance | coverage should be 100% |

You cannot compute a baseline for *"coverage should be 100%"*. There is no
history to average, because it has always been 100% and always should be. That
is a decision, not a statistic, and pretending otherwise is how a slow drift
becomes the new normal.

**And this matters when the agent reads the breach later.** "75% when it is
normally 100%" is ambiguous: is 75 unusual? "75% when a person decided it must be
100%, and anything past 1 is a breach" is not ambiguous. The record has to carry
which kind it is.

## Step 5 · Some numbers are only bad in one direction

`records_held` going **up** is a problem. Going down is a fix. Alerting on a fix
is how people learn to ignore alerts.

In [ ]:
def apply_direction(direction, value, baseline, breached):
    if not breached:
        return False
    if direction == 'above' and value < baseline:  return False
    if direction == 'below' and value > baseline:  return False
    return True

for d, v in [('below', 75.3), ('below', 100.0), ('above', 75.3)]:
    print(f'  watch {d:5}  value {v:6}  vs 100  ->  '
          f'breach={apply_direction(d, v, 100.0, True)}')

## Step 6 · One KPI is a script. Thirteen is a service.

Now we stop writing our own and look at the real catalogue, because the shape is
identical to what we just built, and the interesting part is **which thirteen**.

In [ ]:
from signal_service.kpis import CATALOGUE, BY_NAME, get

groups = {
    'did the work happen': ('pipelines_failing', 'warehouse_lag_hours', 'records_held'),
    'is the volume right': ('rides_per_day', 'revenue_per_day', 'events_per_ride'),
    'is the shape right':  ('completion_rate', 'cancellation_rate', 'avg_fare'),
    'is anything missing': ('surge_coverage_pct', 'fare_coverage_pct',
                            'zone_coverage_pct', 'settlement_coverage_pct'),
}
for question, names in groups.items():
    print(question.upper())
    for name in names:
        k = BY_NAME[name]
        how = 'a rule' if k.judgement == 'fixed' else 'from history'
        print(f'    {k.name:24} {k.owner:14} {how}')
    print()

Four questions, and between them they are what actually goes wrong:

> **did the work happen · is the volume right · is the shape right · is anything missing**

The last group is the interesting one. Those four are the only signals in the
catalogue that can catch today's break, because they are the only ones that ask
*"did everything that should have arrived, arrive"* rather than *"is the number
about right"*.

## Does it catch our break?

In [ ]:
run('cli.py', 'signals')

`surge_coverage_pct` is red. Nothing else moved, because nothing else was
watching for this.

## Step 7 · A breach is a record, not a message

When a signal goes off, something else has to act on it. What we send is the most
consequential design decision in the whole project.

The temptation is a string: *"surge_coverage_pct is 75.33%, was 100%"*. That is
fine for a human and useless to a program: nothing can be branched on, nothing
can be grouped, nothing can be tested.

So the two services share **one typed record**, and it lives in a package that
neither of them owns.

In [ ]:
from events.contract import SignalBreach

print(inspect.getsource(SignalBreach)[:1600])

### Why `events/` is its own package

If the record lived in `signal_service/`, the agent would import from the signal
service, and they would be one program with two folders. The moment the agent
imports the board's code, it can reach past the API into the board's tables, and
nobody notices until a refactor breaks something in a service that "does not
depend on it".

Two services, one contract, neither owns it.

### And it carries how unusual this is

In [ ]:
from signal_service import evaluate as ev

kpi = get('surge_coverage_pct')
reading, verdict = ev.evaluate(kpi)
breach = ev.to_breach(kpi, reading, verdict)

print(breach.one_line())
print()
print('  baseline kind :', breach.baseline_kind)
print('  tolerance     :', breach.tolerance)
print()
print('  ' + breach.how_unusual())

That last sentence is written into the record on purpose. Without it, triage was
dismissing real breaches as ordinary wobble, because "75 when it is normally 100"
does sound like a wobble. **A record that omits how unusual something is invites
the reader to guess.**

## Step 8 · Write it down before you tell anybody

The order of these two lines is the whole reliability story.

In [ ]:
print(inspect.getsource(__import__('signal_service.emit', fromlist=['x'])._dispatch))

```
    record it durably   ->   then ring the doorbell
```

Not the other way round. If the notification fails, or the agent service is
restarting, or the network blinks, the breach is **already in the database** and
`POST /sweep` picks it up later. Cost: some latency. The other order costs you
the incident.

This is the same shape as writing rows before committing a Kafka offset, and it
comes up every time two systems have to agree on something.

## Step 9 · Do not say the same thing every minute

The clock runs every sixty seconds. The break lasts for hours. Without this, one
problem becomes four hundred pages.

In [ ]:
from signal_service import store
print(inspect.getsource(store.already_open))

A **fingerprint** is a stable identity for "this same problem": the KPI plus the
severity. Same fingerprint, still open, seen in the last two hours: record the
reading, say nothing.

## Step 10 · Group before you page

This is the one people skip, and it is why on-call rotations burn out.

A pipeline dies at 3am. `pipelines_failing` goes red. So does
`warehouse_lag_hours`, because nothing ran. So does `rides_per_day`, because gold
is stale. So does `revenue_per_day`, for the same reason.

**Four alerts. One cause.** Send four and the person on call spends their first
ten minutes working out that it is one problem, at 3am, which is exactly when
people are worst at that.

In [ ]:
from signal_service import correlate

print(inspect.getsource(correlate.group))

Four rules, in order:

| | rule | why |
|---|---|---|
| 1 | more than half the board is red | that is the estate, not one number |
| 2 | an upstream signal is red | everything else is downstream of work that did not happen |
| 3 | several signals share an owner | one team, one page |
| 4 | otherwise | one incident each |

Rule 2 is the one worth arguing about. `pipelines_failing`,
`warehouse_lag_hours` and `records_held` describe **whether the work happened**.
Every other KPI describes the result of that work. If the work did not happen,
the results are not independent evidence, they are consequences.

In [ ]:
from signal_service.kpis import BY_NAME

fake = []
for name in ['pipelines_failing', 'rides_per_day', 'revenue_per_day', 'events_per_ride']:
    k = BY_NAME[name]
    r, v = ev.read(k), None
    v = ev.judge(k, r)
    fake.append(ev.to_breach(k, r, v.model_copy(update={'breached': True})))

incidents = correlate.group(fake, board_size=len(BY_NAME))
print(f'{len(fake)} breaches  ->  {len(incidents)} incident')
print()
print(correlate.explain(incidents))

Four in, one out, and the incident names the lead signal and the reason it was
grouped. **The person on call is woken once, and told which of the four to look
at first.**

## Step 11 · The clock

Everything so far is a function. A service is those functions plus something that
calls them on a schedule and never dies.

In [ ]:
run('-m', 'signal_service.scheduler', '--once', '--no-notify')

Three decisions live in that loop, and all three are about what happens on a bad
night:

**It runs OUTSIDE the pipelines.** It cannot block a write and cannot slow a
load. This is the third of three checkpoints, and it is the only one that blocks
nothing:

| checkpoint | where | can it stop you? |
|---|---|---|
| the contract | inside the pipeline, per record | **yes**, it holds the record |
| the invariants | after a write, before committing | **yes**, it rolls back |
| the signal board | outside, on a clock | **no**, and that is the point |

**One cycle never raises.** A cycle that dies takes the clock with it, and a
monitoring system that is down looks exactly like a system with nothing wrong.

**Every reading is saved, breach or not.** Today's readings are tomorrow's
baseline. Throw away the healthy ones and you have no history to be normal
against.

## Step 12 · The API, and why it is over HTTP

The agents could import `signal_service` directly. They talk to it over HTTP
instead.

In [ ]:
from signal_service.api import app

for route in app.routes:
    if hasattr(route, 'methods') and route.path != '/openapi.json':
        print(f'  {",".join(sorted(route.methods)):8} {route.path}')

The boundary is the point. Over HTTP the agent can only ask the questions the API
exposes; with an import it can reach into the board's tables and nobody notices
until a refactor breaks a service that "does not depend on it".

## Step 13 · The other checkpoint: invariants

A signal watches a number that is **usually** in a range. An invariant is a
statement that must **never** be false. There is no tolerance, no baseline and no
history: one violation is one bug.

In [ ]:
run('cli.py', 'invariants')

Every invariant is a query that returns **the rows that violate it**. Zero rows
is a pass. That shape is deliberate: when it fails you already have the evidence
in your hand rather than a boolean and a hunt.

> **A signal that fires is a question for a human.**
> **An invariant that fires is a defect.**

---

# What you built, and where it actually lives

Everything in Part 2 is in `signal_service/`, in the same order you built it.

| file | what it is | the step |
|---|---|---|
| `kpis.py` | the thirteen definitions | 2, 6 |
| `evaluate.py` | `read` and `judge`, kept apart | 3, 4, 5 |
| `invariants.py` | the seventeen things that must never be false | 13 |
| `store.py` | readings, breaches, incidents, suppression | 8, 9 |
| `correlate.py` | four breaches into one incident | 10 |
| `emit.py` | record durably, then ring the doorbell | 8 |
| `scheduler.py` | the clock, which never dies | 11 |
| `api.py` | the questions other services may ask | 12 |
| `dashboard.py` | all of it in a browser | |
| `events/contract.py` | the record both services share | 7 |

Every file opens with a docstring explaining the **decision**, not the syntax.

## See all of it at once

In [ ]:
run('cli.py', 'signals')
run('cli.py', 'incidents')

And in a browser, on a second screen, refreshing itself. In a terminal:

```
python cli.py dashboard          # then open http://localhost:8099
```

Click any tile to see the exact query behind it.

> **The dashboard is not the system.** The system is thirteen queries, a baseline
> for each, and a clock. The dashboard is a nice way to read the answer.

---

# What to take away

> ### A number cannot raise an alarm. Only a number plus somebody's written opinion of what normal looks like.
> ### Writing that opinion down, with an owner's name on it, is the work.
> ### Group before you page. One cause producing four alerts is not four problems.
> ### Record it durably, then notify. Never the other way round.

**Next:** the board has found the problem and told nobody what caused it. That is
notebook 2.

## Put it back

Clears the break, the incidents and the artifacts, so this notebook is ready to
run again from the top.

In [ ]:
run('break_it.py', '--fix')     # undo the break, if it is still in place
run('cli.py', 'reset')          # empty warehouse, no incidents, no artifacts
run('cli.py', 'run', 'all')     # rebuild all eight pipelines, about 40 seconds